# **Maestría en Inteligencia Artificial Aplicada**

## Curso: **Procesamiento de Lenguaje Natural**

### Tecnológico de Monterrey

### Prof Luis Eduardo Falcón Morales

### **Actividad en Equipo - Semanas 4 y 5**

### **Vectores Embebidos de HuggingFace**

#### **Nombres y matrículas de los integrantes del equipo:**



*   Elemento de lista
*   Elemento de lista



In [1]:
# Aquí deberán incluir todas las librerías que requieran durante esta actividad:
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
import re
import pickle


En esta actividad deberás utilizar los datos de tres archivos que se encuentran en el repositorio de la UCI llamados **amazon_cells_labelled.txt**, **imdb_labelled.txt** y   **yelp_labelled.txt**. Cada uno de estos archivos corresponden a comentarios de usuarios que adquirieron un celular a través de la plataforma de Amazon, de comentarios que dejaron usuarios sobre palículas y series en la plataforma de IMDb y sobre servicios de comida dejados en la plataforma de Yelp.

La información del problema y de los archivos están basados en el repositorio de la UCI cuya liga es la siguiente:

https://archive.ics.uci.edu/dataset/331/sentiment+labelled+sentences



# **Pregunta - 1:**



Descarga los 3 archivos de la plataforma de la UCI indicado previamente y genera un nuevo DataFrame de Pandas con ellos.

**Llama simplemente "df" a dicho DataFrame.**




In [2]:

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********
nltk.download('stopwords')    # para tener acceso a "stopwords" en varios idiomas.

dfa = pd.read_csv('/content/sample_data/amazon_cells_labelled.txt', sep='\t', names=['review','label'], header=None, encoding='utf-8')
dfi = pd.read_csv('/content/sample_data/imdb_labelled.txt', sep=r'[\t|\n]', names=['review','label'], header=None, encoding='utf-8')
dfy = pd.read_csv('/content/sample_data/yelp_labelled.txt', sep='\t', names=['review','label'], header=None, encoding='utf-8')


df = pd.concat([dfa, dfi, dfy], ignore_index=True)



# *********** Aquí termina la sección de agregar código *************


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
/tmp/ipykernel_3011/856491289.py:5: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  dfi = pd.read_csv('/content/sample_data/imdb_labelled.txt', sep=r'[\t|\n]', names=['review','label'], header=None, encoding='utf-8')


In [3]:
# Verifiquemos la información del DataFrame:

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   review  3000 non-null   object
 1   label   3000 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 47.0+ KB


In [4]:
# Y mostremos sus primeros registros:

df.head()

,review,label
0,So there is no way for me to plug it in here i...,0
1,"Good case, Excellent value.",1
2,Great for the jawbone.,1
3,Tied to charger for conversations lasting more...,0
4,The mic is great.,1


# **Pregunta - 2:**

Proceso de limpieza. Aplica el proceso de limpieza que consideres adecuado.











In [5]:

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********
negwords = [ 'no', 'nor', 'not', 'ain', 'aren', "aren't", 'don', "don't", 'couldn', "couldn't", 'didn', "didn't", 'doesn', "doesn't", 'hadn', "hadn't", 'hasn', "hasn't", 'haven', "haven't", 'isn', "isn't", 'mightn', "mightn't", 'mustn', "mustn't", 'needn', "needn't", 'shan', "shan't", 'shouldn', "shouldn't", 'wasn', "wasn't", 'weren', "weren't", 'won', "won't", 'wouldn', "wouldn't"]
posWords = ["good","happy","joyful","excellent","amazing","fantastic","great","awesome","positive","fortunate","pleasant","delightful","brilliant","outstanding","superb","wonderful","lovely","cheerful","optimistic","successful","vibrant"]

def clean_tok(position, doc):
  textMin = str.lower(doc)
  mystopwords = [w for w in stopwords.words('english') if w not in negwords]
  import re
  patron=r'\b[a-z]{2,}\b'
  tokens = re.findall(patron,textMin) # Tomar solo palabras alfabeticas
  tokens = [re.sub(r'(.)\1{2,}', r'\1', w) for w in tokens if w not in mystopwords]
  for i,t in enumerate(tokens):
    if t in posWords: # Verificar que no existan casos como not good, cambiando palabra good por notGood
      try:
        if tokens[i-1] in negwords:
          tokens[i] = 'not'+t
      except:
        pass
    if t == "": # Rellenando filas sin datos solo creando una etiqueta generica basada en la etiqueta
      if df['label'][position] == 0:
        tokens[i] = 'bad'
      else:
        tokens[i] = 'good'

  return tokens

Xclean = [ clean_tok(i,doc) for i,doc in enumerate(df['review'])]

# *********** Aquí termina la sección de agregar código *************

In [6]:
# Despleguemos los primeros comentarios después de tu proceso de limpieza:

for x in Xclean[0:5]:
  print(x)


['no', 'way', 'plug', 'us', 'unless', 'go', 'converter']
['good', 'case', 'excellent', 'value']
['great', 'jawbone']
['tied', 'charger', 'conversations', 'lasting', 'minutes', 'major', 'problems']
['mic', 'great']


# **Pregunta - 3:**



Realicemos una partición aleatoria con los porcentajes que consideres más adecuados. Utiliza una semilla para su reproducibilidad.

In [7]:

# ************* Inicia la sección de agregar código:*****************************


from sklearn.model_selection import train_test_split

y = df['label']
Xtrain, Xval_test, ytrain, yval_test = train_test_split(Xclean, y, train_size=.70, shuffle=True, random_state=43)
Xval, Xtest, yval, ytest = train_test_split(Xval_test, yval_test, test_size=.50, shuffle=True, random_state=43)




# *********** Termina la sección de agregar código *************


# verificemos las dimensiones obtenidas:
print('X,y Train:', len(Xtrain), len(ytrain))
print('X,y Val:', len(Xval), len(yval))
print('X,y Test', len(Xtest), len(ytest))

X,y Train: 2100 2100
X,y Val: 450 450
X,y Test 450 450


# **Pregunta - 4:**




### **Construye tu vocabulario a continuación utilizando solamente el conjunto de Train:**


In [8]:
# a.	Usa el conjunto de entrenamiento para generar tu vocabulario
#     con un tamaño que consideres adecuado:


# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********
from collections import Counter

midiccionario = Counter()

for k in range(len(Xtrain)):
  midiccionario.update(Xtrain[k])

min_freq = 1

midicc = {}
for k,v in midiccionario.items():
  if v > min_freq:
    midicc[k] = v

print('Longitud del diccionario:', len(midiccionario))

# *********** Aquí termina la sección de agregar código *************




Longitud del diccionario: 3961


In [9]:
# b.	Indica el tamaño del vocabulario generado.

print('Longitud del vocabulario generado:')


# ******* Inicia la sección de agregar código: ***********


print(len(midicc))




# *********** Aquí termina la sección de agregar código *************

Longitud del vocabulario generado:
1601


In [10]:
# c.	Con el vocabulario generado, filtra los conjuntos de entrenamiento,
#     validación y prueba para que todos los comentarios usen solamente las
#     palabras de este vocabulario.

#     Llamar train_X, val_X y test_X a estos tres conjuntos.


# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********


def filtrar_vocabulario(dataset, vocabulario, y):
  resultado = []
  retVal = []
  for i,token in enumerate(dataset):
    for t in token:
      if t in vocabulario:
        resultado.append(t)
    if token == []:
      if y[i] == 0:
        resultado.append('bad')
      else:
        resultado.append('good')
    retVal.append(resultado)
    resultado = []

  return retVal

tokensList = list(midicc.keys())

# Filtrar datasets
train_X = filtrar_vocabulario(Xtrain, tokensList, ytrain.to_list())
val_X   = filtrar_vocabulario(Xval, tokensList, yval.to_list())
test_X  = filtrar_vocabulario(Xtest, tokensList, ytest.to_list())



# *********** Aquí termina la sección de agregar código *************


In [11]:
# Vemos el resultado de los primeros comentarios del conjunto de validación:

for ss in val_X[0:5]:
  print(ss)

['seller', 'would', 'definitely', 'buy']
['lacking']
['internet', 'slow']
['phone', 'sturdy']
['not', 'work', 'time', 'nokia']


# **Pregunta - 5:**

Incluye tus comentarios sobre cada modelo de HuggingFace indicado.

### ++++++++ Inicia la sección de agregar texto: +++++++++++

* **a) bge-base-en-v1.5**
* ** No son comentarios aun**
>model that transforms any given text into a 768-dimensional vector.a highly popular, compact English text embedding model developed by BAAI. Designed for Retrieval-Augmented Generation (RAG) and semantic search, it maps text to 768-dimensional vectors and significantly improves upon the original v1 model by providing a more natural similarity distribution and better retrieval capabilities without needing explicit prompts.

* **b) bge-large-en-v1.5**
* ** No son comentarios aun**
 >Modelo de embeddings de alto rendimiento centrado en inglés que transforma el texto en representaciones vectoriales precisas de 1024 dimensiones, logrando una impresionante puntuación MTEB de 64.23. Este modelo destaca en la búsqueda semántica, la recuperación de documentos, la comparación de similitud y aplicaciones de clustering, lo que lo hace ideal para construir motores de búsqueda sofisticados, sistemas de recomendación y plataformas de gestión del conocimiento donde la comprensión, recuperación e indexación precisa de texto en inglés son críticas.

None

* **c)	e5-base-v2**
* ** No son comentarios aun**
>is a highly efficient open-source text embedding model that maps sentences and passages into a 768-dimensional dense vector space. Highly regarded on the MTEB leaderboard, it excels at semantic search, Retrieval-Augmented Generation (RAG), and clustering

None

### ++++++++ Termina la sección de agregar texto: +++++++++++

# **Pregunta - 6:**

In [12]:
# a) Cargar el modelo de embeddings de HuggingFace seleccionado:

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('BAAI/bge-base-en-v1.5')

embeddings_1 = model.encode([f"query: {p}" for p in list(midicc.keys())], normalize_embeddings=True)


# *********** Aquí termina la sección de agregar código *************

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [13]:
#Export
pickle.dump(embeddings_1, open('/content/sample_data/embeddings_base_052726.pkl', 'wb'))

In [14]:
#Import
#embeddings_1 = pickle.load(open('/content/sample_data/embeddings_1.pkl', 'rb'))

In [15]:
# b) Primeros 3 elementos clave:valor del diccionario generado.

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

embDict = {
    palabra: vector.tolist()
    for palabra, vector in zip(list(midicc.keys()), embeddings_1)
}

sel = 3
for k,v in embDict.items():
  print(k,v)
  sel -= 1
  if sel == 0:
    break

# *********** Aquí termina la sección de agregar código *************



meat [0.024973316118121147, 0.009044612757861614, -0.031490977853536606, 0.0015152029227465391, 0.03841531649231911, 0.0236501544713974, 0.0046532489359378815, 0.01606440171599388, -0.018235960975289345, -0.01732162944972515, 0.0056660654954612255, 0.012679174542427063, -0.01555208396166563, -0.017055967822670937, 0.006438718643039465, 0.06828442215919495, 0.03122343122959137, 0.022524649277329445, -0.013947103172540665, -0.010805102996528149, 0.004906815942376852, -0.04312177374958992, -0.011965653859078884, 0.026410197839140892, 0.046706411987543106, -0.010869676247239113, 0.0076461960561573505, 0.03042689710855484, -0.06069611757993698, -0.025189140811562538, 0.011491945944726467, 0.007456374354660511, 0.024227028712630272, -0.03484288603067398, 0.005361090414226055, -0.03003588132560253, 0.0016626645810902119, 0.005173185840249062, -0.0012402139836922288, -5.4479129175888374e-05, -0.06384788453578949, 0.0326390378177166, 0.01838499866425991, 0.00850729551166296, -0.0185630414634943

In [16]:
# c) Tamaño del diccionario generado:

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********


print('Tamaño del diccionario generado:', len(embDict))


# *********** Aquí termina la sección de agregar código *************


Tamaño del diccionario generado: 1601


# **Pregunta - 7:**




Generamos los vectores embebidos a partir de los conjuntos de entrenamiento, validación y prueba y con las características indicadas en el archivo PDF.

Los llamaremos trainEmb, valEmb y testEmb, respectivamente.


In [17]:
# a) Comentarios con vectores embebidos.

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

def PromedioEmb(DictEmb,Frase):
  arrPrmdio = np.array(np.zeros(768))
  for p in Frase:
    arrPrmdio = arrPrmdio + np.array(DictEmb[p])
  return arrPrmdio/768

trainEmb = [PromedioEmb(embDict,doc) for doc in train_X]
valEmb = [PromedioEmb(embDict,doc) for doc in val_X]
testEmb = [PromedioEmb(embDict,doc) for doc in test_X]


# *********** Aquí termina la sección de agregar código *************

In [18]:
# b) Dimensiones de los conjuntos trainEmb, valEmb y testEmb.

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

print('Dimensiones de los conjuntos trainEmb, valEmb y testEmb:')
print('trainEmb:', np.array(trainEmb).shape)
print('valEmb:', np.array(valEmb).shape)
print('testEmb:', np.array(testEmb).shape)


# *********** Aquí termina la sección de agregar código *************

Dimensiones de los conjuntos trainEmb, valEmb y testEmb:
trainEmb: (2100, 768)
valEmb: (450, 768)
testEmb: (450, 768)


# **Pregunta - 8:**

In [19]:
# Número de tokens generedos al obtener cada uno de los conjuntos trainEmb, valEmb y testEmb.

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********


trainEmbTokens = [len(doc) for doc in train_X]
valEmbTokens = [len(doc) for doc in val_X]
testEmbTokens = [len(doc) for doc in test_X]

print('Número de tokens generedos al obtener cada uno de los conjuntos trainEmb, valEmb y testEmb:')
print('trainEmb:', np.array(trainEmbTokens).shape)
# *********** Aquí termina la sección de agregar código *************

Número de tokens generedos al obtener cada uno de los conjuntos trainEmb, valEmb y testEmb:
trainEmb: (2100,)


# **Pregunta - 9:**



Entrenamiento y reporte de los modelos de Regresión Logística y Bosque Aleatorio (Random Forest).


In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

In [25]:
# 9a) REGRESIÓN LOGÍSTICA:

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

# Modelo de Regresión Logística (Logistic Regression):
modeloLR = LogisticRegression(random_state=43, max_iter=3000, C=5)
modeloLR.fit(trainEmb, ytrain)

print(classification_report(yval, modeloLR.predict(valEmb)))
print(classification_report(ytrain, modeloLR.predict(trainEmb)))
# *********** Aquí termina la sección de agregar código *************


              precision    recall  f1-score   support

           0       0.47      1.00      0.64       212
           1       0.00      0.00      0.00       238

    accuracy                           0.47       450
   macro avg       0.24      0.50      0.32       450
weighted avg       0.22      0.47      0.30       450

              precision    recall  f1-score   support

           0       0.50      1.00      0.67      1055
           1       0.00      0.00      0.00      1045

    accuracy                           0.50      2100
   macro avg       0.25      0.50      0.33      2100
weighted avg       0.25      0.50      0.34      2100



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

In [23]:
# 9b) BOSQUE ALEATORIO (Random Forest):

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********


modeloRF = RandomForestClassifier(random_state=43, n_estimators=300, max_depth=30, min_samples_leaf=2,min_samples_split=3)
modeloRF.fit(trainEmb, ytrain)

print(classification_report(yval, modeloRF.predict(valEmb)))
print(classification_report(ytrain, modeloRF.predict(trainEmb)))

# *********** Aquí termina la sección de agregar código *************

              precision    recall  f1-score   support

           0       0.77      0.83      0.80       212
           1       0.84      0.78      0.81       238

    accuracy                           0.80       450
   macro avg       0.81      0.81      0.80       450
weighted avg       0.81      0.80      0.80       450

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1055
           1       1.00      1.00      1.00      1045

    accuracy                           1.00      2100
   macro avg       1.00      1.00      1.00      2100
weighted avg       1.00      1.00      1.00      2100



# **Pregunta - 10**

**Proceso basado en modelos Preentrenados**

In [ ]:
# 10a) Partición.:

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********


None


# *********** Aquí termina la sección de agregar código *************

In [ ]:
# 10b) Vectores embebidos:

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********


None


# *********** Aquí termina la sección de agregar código *************

In [ ]:
# 10c) REGRESIÓN LOGÍSTICA.

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********


None


# *********** Aquí termina la sección de agregar código *************

In [ ]:
# 10d) BOSQUE ALEATORIO.

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********


None


# *********** Aquí termina la sección de agregar código *************

# **Pregunta - 11:**

In [ ]:
# Reporte del mejor modelo y partición con el conjunto de Prueba.

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********


None


# *********** Aquí termina la sección de agregar código *************

# **Pregunta - 12:**



Incluye tus comentarios finales de la actividad.

### ++++++++ Inicia la sección de agregar texto: +++++++++++

None

### ++++++++ Termina la sección de agregar texto: +++++++++++

# **Fin de la Actividad de Vectores Embebidos - HuggingFace**